# Traffic Crash Data Analysis

#### Import required libraries

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
from sqlalchemy import create_engine, text
import sqlite3

#### Phase 1: Data loading and understanding

In [ ]:
# Mounting drive to colab
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Load CSV file to colab from google drive
df = pd.read_csv("/content/drive/MyDrive/Data Science Program - HCL GUVI/Traffic Crash Analytics/Traffic_CrashesData.csv")

In [ ]:
#see all column names
df.columns.tolist()

['CRASH_RECORD_ID',
 'CRASH_DATE',
 'POSTED_SPEED_LIMIT',
 'TRAFFIC_CONTROL_DEVICE',
 'DEVICE_CONDITION',
 'WEATHER_CONDITION',
 'LIGHTING_CONDITION',
 'FIRST_CRASH_TYPE',
 'TRAFFICWAY_TYPE',
 'ALIGNMENT',
 'ROADWAY_SURFACE_COND',
 'ROAD_DEFECT',
 'REPORT_TYPE',
 'CRASH_TYPE',
 'DAMAGE',
 'DATE_POLICE_NOTIFIED',
 'PRIM_CONTRIBUTORY_CAUSE',
 'SEC_CONTRIBUTORY_CAUSE',
 'STREET_NO',
 'STREET_DIRECTION',
 'STREET_NAME',
 'BEAT_OF_OCCURRENCE',
 'NUM_UNITS',
 'MOST_SEVERE_INJURY',
 'INJURIES_TOTAL',
 'INJURIES_FATAL',
 'INJURIES_INCAPACITATING',
 'INJURIES_NON_INCAPACITATING',
 'INJURIES_REPORTED_NOT_EVIDENT',
 'INJURIES_NO_INDICATION',
 'INJURIES_UNKNOWN',
 'CRASH_HOUR',
 'CRASH_DAY_OF_WEEK',
 'CRASH_MONTH',
 'LATITUDE',
 'LONGITUDE',
 'LOCATION',
 'date',
 'year']

In [ ]:
#see the data types of every column
df.dtypes

,0
CRASH_RECORD_ID,object
CRASH_DATE,object
POSTED_SPEED_LIMIT,int64
TRAFFIC_CONTROL_DEVICE,object
DEVICE_CONDITION,object
WEATHER_CONDITION,object
LIGHTING_CONDITION,object
FIRST_CRASH_TYPE,object
TRAFFICWAY_TYPE,object
ALIGNMENT,object


In [ ]:
#how many rows and columns
print(f"Rows: {len(df)}")
print(f"Columns: {df.shape[1]}")

Rows: 660934
Columns: 39


In [ ]:
#see the first 5 rows
df.head()
#see the last 5 rows
df.tail()

,CRASH_RECORD_ID,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,ALIGNMENT,...,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION,date,year
660929,5ff5b050ae2079df6addca8431c1bd6e381432ff72ff43...,10/18/2025 11:42:00 AM,35,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLOUDY/OVERCAST,DAYLIGHT,REAR END,NOT DIVIDED,STRAIGHT AND LEVEL,...,1.0,0.0,11,7,10,41.938537,-87.835965,POINT (-87.835965097138 41.938536628877),2025-10-18 11:42:00,2025
660930,7a7963ac15a38f3cf7549397dd053fd7612dd5194de280...,10/18/2025 03:50:00 AM,35,NO CONTROLS,NO CONTROLS,RAIN,DARKNESS,FIXED OBJECT,NOT DIVIDED,STRAIGHT AND LEVEL,...,1.0,0.0,3,7,10,41.899200,-87.618841,POINT (-87.618840672611 41.8992002546),2025-10-18 03:50:00,2025
660931,84e546c12fd8fd78ae4e2dc26fdb02d6e8f6c4c92be450...,10/18/2025 05:08:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,SIDESWIPE SAME DIRECTION,NOT DIVIDED,STRAIGHT AND LEVEL,...,5.0,0.0,17,7,10,41.848081,-87.675881,POINT (-87.675881338024 41.848080595588),2025-10-18 17:08:00,2025
660932,8237045757e7fed5adfb450bd7c8472e28a61d322f86bb...,10/19/2025 01:50:00 AM,30,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,REAR TO FRONT,NOT DIVIDED,STRAIGHT AND LEVEL,...,5.0,0.0,1,1,10,41.949157,-87.648527,POINT (-87.648526617888 41.949157243366),2025-10-19 01:50:00,2025
660933,ede74fa6fd507334cd0415d3d53deb535d27d23523cd38...,06/23/2023 04:30:00 PM,30,UNKNOWN,UNKNOWN,UNKNOWN,DAYLIGHT,REAR END,NOT DIVIDED,STRAIGHT AND LEVEL,...,1.0,0.0,16,6,6,42.004564,-87.709200,POINT (-87.709200383599 42.004564031467),2023-06-23 16:30:00,2023


In [ ]:
#see basic statistics for numerical columns
df.describe()

,POSTED_SPEED_LIMIT,STREET_NO,BEAT_OF_OCCURRENCE,NUM_UNITS,INJURIES_TOTAL,INJURIES_FATAL,INJURIES_INCAPACITATING,INJURIES_NON_INCAPACITATING,INJURIES_REPORTED_NOT_EVIDENT,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,year
count,660934.000000,660934.000000,660934.000000,660934.000000,660934.000000,660934.000000,660934.000000,660934.000000,660934.000000,660934.000000,660934.0,660934.000000,660934.000000,660934.000000,660934.000000,660934.000000,660934.000000
mean,28.550056,3755.664555,1247.928255,2.039434,0.218258,0.003602,0.021922,0.122596,0.077105,1.964009,0.0,13.189364,4.117346,6.434270,41.853631,-87.673909,2022.705571
std,5.580016,2837.249372,701.841337,0.464771,0.602016,0.061753,0.169913,0.447240,0.354696,1.129489,0.0,5.634005,1.984167,3.427161,0.352360,0.717845,1.772997
min,0.000000,1.000000,111.000000,1.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,1.000000,1.000000,0.000000,-87.939678,2020.000000
25%,30.000000,1398.000000,723.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.0,9.000000,2.000000,3.000000,41.780931,-87.722543,2021.000000
50%,30.000000,3300.000000,1135.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.0,14.000000,4.000000,6.000000,41.873273,-87.675638,2023.000000
75%,30.000000,5600.000000,1821.000000,2.000000,0.000000,0.000000,0.000000,0.000000,0.000000,2.000000,0.0,17.000000,6.000000,9.000000,41.924714,-87.634545,2024.000000
max,70.000000,13799.000000,2535.000000,18.000000,21.000000,4.000000,10.000000,19.000000,19.000000,49.000000,0.0,23.000000,7.000000,12.000000,42.022780,0.000000,2026.000000


In [ ]:
#check for missing values
df.isnull().sum()

,0
CRASH_RECORD_ID,0
CRASH_DATE,0
POSTED_SPEED_LIMIT,0
TRAFFIC_CONTROL_DEVICE,0
DEVICE_CONDITION,0
WEATHER_CONDITION,0
LIGHTING_CONDITION,0
FIRST_CRASH_TYPE,0
TRAFFICWAY_TYPE,0
ALIGNMENT,0


#### Connect to Database
    Create a SQL database and load the data

In [ ]:
# Create SQLite database
engine = create_engine('sqlite:///traffic_crashes.db')

# Load dataframe into the database as a table
df.to_sql('CrashTable', con=engine, if_exists='replace', index=False)

print("Connected!")
print("Table 'CrashTable' loaded with", len(df), "records")

Connected!
Table 'CrashTable' loaded with 660934 records


#### Phase 2: Data exploration
    Execute basic SQL queries against the Traffic Crash table

In [ ]:
# Execute SQL query and get results
query = "SELECT * FROM CrashTable"
pd.read_sql(query, con=engine)

,CRASH_RECORD_ID,CRASH_DATE,POSTED_SPEED_LIMIT,TRAFFIC_CONTROL_DEVICE,DEVICE_CONDITION,WEATHER_CONDITION,LIGHTING_CONDITION,FIRST_CRASH_TYPE,TRAFFICWAY_TYPE,ALIGNMENT,...,INJURIES_NO_INDICATION,INJURIES_UNKNOWN,CRASH_HOUR,CRASH_DAY_OF_WEEK,CRASH_MONTH,LATITUDE,LONGITUDE,LOCATION,date,year
0,000c4307d8e9b39075cffdd0aade3603e0f96f14e41da9...,01/14/2025 12:25:00 PM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,SIDESWIPE SAME DIRECTION,DIVIDED - W/MEDIAN (NOT RAISED),STRAIGHT AND LEVEL,...,2.0,0.0,12,3,1,41.997808,-87.655770,POINT (-87.655770494712 41.997807727633),2025-01-14 12:25:00,2025
1,027b0b4c21460d3441fd83929abb9673c6fc0c7d575675...,05/23/2025 09:30:00 AM,30,STOP SIGN/FLASHER,UNKNOWN,UNKNOWN,DAYLIGHT,TURNING,DIVIDED - W/MEDIAN (NOT RAISED),STRAIGHT AND LEVEL,...,2.0,0.0,9,6,5,41.946529,-87.688106,POINT (-87.688106391039 41.946529480518),2025-05-23 09:30:00,2025
2,04d91dffc94f677358ca47056921ba5c4224320df27ed4...,04/05/2025 08:00:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,UNKNOWN,PARKED MOTOR VEHICLE,NOT DIVIDED,STRAIGHT AND LEVEL,...,1.0,0.0,20,7,4,41.899325,-87.715074,POINT (-87.715074373867 41.899324573751),2025-04-05 20:00:00,2025
3,0b5603954d84b7341c7cad4f570ea039e85919f3750ccb...,05/23/2025 09:15:00 AM,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,PEDALCYCLIST,NOT DIVIDED,STRAIGHT AND LEVEL,...,2.0,0.0,9,6,5,41.902793,-87.699412,POINT (-87.699412181285 41.902792968177),2025-05-23 09:15:00,2025
4,00bce77960c2faa2a8782a8cac1d6e5715802d6c072a08...,01/14/2025 08:00:00 AM,30,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,SNOW,DAYLIGHT,REAR END,FOUR WAY,STRAIGHT AND LEVEL,...,2.0,0.0,8,3,1,41.691207,-87.720555,POINT (-87.720554863466 41.691206664451),2025-01-14 08:00:00,2025
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
660929,5ff5b050ae2079df6addca8431c1bd6e381432ff72ff43...,10/18/2025 11:42:00 AM,35,TRAFFIC SIGNAL,FUNCTIONING PROPERLY,CLOUDY/OVERCAST,DAYLIGHT,REAR END,NOT DIVIDED,STRAIGHT AND LEVEL,...,1.0,0.0,11,7,10,41.938537,-87.835965,POINT (-87.835965097138 41.938536628877),2025-10-18 11:42:00,2025
660930,7a7963ac15a38f3cf7549397dd053fd7612dd5194de280...,10/18/2025 03:50:00 AM,35,NO CONTROLS,NO CONTROLS,RAIN,DARKNESS,FIXED OBJECT,NOT DIVIDED,STRAIGHT AND LEVEL,...,1.0,0.0,3,7,10,41.899200,-87.618841,POINT (-87.618840672611 41.8992002546),2025-10-18 03:50:00,2025
660931,84e546c12fd8fd78ae4e2dc26fdb02d6e8f6c4c92be450...,10/18/2025 05:08:00 PM,30,NO CONTROLS,NO CONTROLS,CLEAR,DAYLIGHT,SIDESWIPE SAME DIRECTION,NOT DIVIDED,STRAIGHT AND LEVEL,...,5.0,0.0,17,7,10,41.848081,-87.675881,POINT (-87.675881338024 41.848080595588),2025-10-18 17:08:00,2025
660932,8237045757e7fed5adfb450bd7c8472e28a61d322f86bb...,10/19/2025 01:50:00 AM,30,UNKNOWN,UNKNOWN,UNKNOWN,UNKNOWN,REAR TO FRONT,NOT DIVIDED,STRAIGHT AND LEVEL,...,5.0,0.0,1,1,10,41.949157,-87.648527,POINT (-87.648526617888 41.949157243366),2025-10-19 01:50:00,2025


#### Phase 3: Advanced SQL Analysis

Q1: Top 5 most dangerous combinations of weather and crash type

In [ ]:
q1 = pd.read_sql("""
    SELECT
        WEATHER_CONDITION,
        FIRST_CRASH_TYPE,
        COUNT(*) AS total_crashes
    FROM CrashTable
    GROUP BY WEATHER_CONDITION, FIRST_CRASH_TYPE
    ORDER BY total_crashes DESC
    LIMIT 5
""", engine)

q1

,WEATHER_CONDITION,FIRST_CRASH_TYPE,total_crashes
0,CLEAR,PARKED MOTOR VEHICLE,117073
1,CLEAR,REAR END,102846
2,CLEAR,SIDESWIPE SAME DIRECTION,80197
3,CLEAR,TURNING,78528
4,CLEAR,ANGLE,58317


Q2: Top 10 streets with the highest number of injury crashes

In [ ]:
q2 = pd.read_sql("""
    SELECT
        STREET_NAME,
        COUNT(*) AS injury_crashes
    FROM CrashTable
    WHERE INJURIES_TOTAL > 0
    GROUP BY STREET_NAME
    ORDER BY injury_crashes DESC
    LIMIT 10
""", engine)

q2

,STREET_NAME,injury_crashes
0,WESTERN AVE,2973
1,PULASKI RD,2840
2,ASHLAND AVE,2579
3,CICERO AVE,2560
4,HALSTED ST,2536
5,KEDZIE AVE,1965
6,STATE ST,1379
7,MICHIGAN AVE,1344
8,STONY ISLAND AVE,1340
9,DAMEN AVE,1263


Q3: Percentage of crashes that resulted in injuries for each crash type

In [ ]:
q3 = pd.read_sql("""
    SELECT
        FIRST_CRASH_TYPE,
        COUNT(*) AS total_crashes,
        SUM(CASE WHEN INJURIES_TOTAL > 0 THEN 1 ELSE 0 END) AS injury_crashes,
        ROUND(
            100.0 * SUM(CASE WHEN INJURIES_TOTAL > 0 THEN 1 ELSE 0 END) / COUNT(*),
        2) AS injury_percentage
    FROM CrashTable
    GROUP BY FIRST_CRASH_TYPE
    ORDER BY injury_percentage DESC
""", engine)

q3

,FIRST_CRASH_TYPE,total_crashes,injury_crashes,injury_percentage
0,PEDESTRIAN,16127,14230,88.24
1,PEDALCYCLIST,11522,8355,72.51
2,TRAIN,39,20,51.28
3,OVERTURNED,421,194,46.08
4,HEAD ON,5561,2057,36.99
5,ANGLE,73576,18766,25.51
6,OTHER NONCOLLISION,1633,410,25.11
7,OTHER OBJECT,6960,1435,20.62
8,FIXED OBJECT,31306,6354,20.30
9,TURNING,97571,17787,18.23


Q4: Peak crash hour for each month

In [ ]:
q4 = pd.read_sql("""
    WITH hourly_counts AS (
        SELECT
            CRASH_MONTH,
            CRASH_HOUR,
            COUNT(*) AS crash_count,
            RANK() OVER (PARTITION BY CRASH_MONTH ORDER BY COUNT(*) DESC) AS rnk
        FROM CrashTable
        WHERE CRASH_MONTH IS NOT NULL
          AND CRASH_HOUR IS NOT NULL
        GROUP BY CRASH_MONTH, CRASH_HOUR
    )
    SELECT CRASH_MONTH, CRASH_HOUR, crash_count
    FROM hourly_counts
    WHERE rnk = 1
    ORDER BY CRASH_MONTH
""", engine)

q4

,CRASH_MONTH,CRASH_HOUR,crash_count
0,1,15,4408
1,2,15,4525
2,3,15,4610
3,4,15,3884
4,5,15,4727
5,6,16,4552
6,7,16,4299
7,8,16,4504
8,9,15,4597
9,10,15,4644


Q5: Top 5 primary causes of nighttime crashes

In [ ]:
q5 = pd.read_sql("""
    SELECT
        PRIM_CONTRIBUTORY_CAUSE,
        COUNT(*) AS crash_count
    FROM CrashTable
    WHERE CRASH_HOUR >= 18
    GROUP BY PRIM_CONTRIBUTORY_CAUSE
    ORDER BY crash_count DESC
    LIMIT 5
""", engine)

q5

,PRIM_CONTRIBUTORY_CAUSE,crash_count
0,UNABLE TO DETERMINE,63606
1,FAILING TO YIELD RIGHT-OF-WAY,16213
2,FOLLOWING TOO CLOSELY,12070
3,NOT APPLICABLE,7829
4,IMPROPER OVERTAKING/PASSING,7710


Q6: Average number of injuries in daylight vs darkness conditions

In [ ]:
q6 = pd.read_sql("""
    SELECT
        LIGHTING_CONDITION,
        ROUND(AVG(INJURIES_TOTAL), 3) AS avg_injuries
    FROM CrashTable
    WHERE LIGHTING_CONDITION IS NOT NULL
    AND LIGHTING_CONDITION IN ('DAYLIGHT', 'DARKNESS', 'DARKNESS, LIGHTED ROAD')
    GROUP BY LIGHTING_CONDITION
    ORDER BY avg_injuries DESC
""", engine)

q6

,LIGHTING_CONDITION,avg_injuries
0,"DARKNESS, LIGHTED ROAD",0.286
1,DARKNESS,0.217
2,DAYLIGHT,0.207


Q7: Traffic control device type with the highest average injuries per crash

In [ ]:
q7 = pd.read_sql("""
    SELECT
        TRAFFIC_CONTROL_DEVICE,
        ROUND(AVG(INJURIES_TOTAL), 3) AS avg_injuries
    FROM CrashTable
    WHERE TRAFFIC_CONTROL_DEVICE IS NOT NULL
    GROUP BY TRAFFIC_CONTROL_DEVICE
    ORDER BY avg_injuries DESC
    LIMIT 10
    """, engine)
q7

,TRAFFIC_CONTROL_DEVICE,avg_injuries
0,BICYCLE CROSSING SIGN,0.655
1,PEDESTRIAN CROSSING SIGN,0.623
2,FLASHING CONTROL SIGNAL,0.446
3,YIELD,0.352
4,NO PASSING,0.351
5,STOP SIGN/FLASHER,0.337
6,TRAFFIC SIGNAL,0.313
7,OTHER RAILROAD CROSSING,0.308
8,DELINEATORS,0.297
9,POLICE/FLAGMAN,0.275


Q8: Top 5 locations (latitude/longitude) with the highest crash frequency

In [ ]:
q8 = pd.read_sql("""
    SELECT
        ROUND(LATITUDE, 4) AS lat,
        ROUND(LONGITUDE, 4) AS lon,
        COUNT(*) AS crash_count
    FROM CrashTable
    WHERE LATITUDE IS NOT NULL AND LONGITUDE IS NOT NULL
    GROUP BY lat, lon
    ORDER BY crash_count DESC
    LIMIT 5
""", engine)

q8

,lat,lon,crash_count
0,41.9762,-87.9053,1247
1,41.9010,-87.6199,665
2,41.7914,-87.5801,479
3,41.7515,-87.5860,469
4,41.7223,-87.5853,353


Q9: Top 5 streets with the highest injury rate (considering only streets with more than 100 crashes)

In [ ]:
q9 = pd.read_sql("""
    SELECT
        STREET_NAME,
        COUNT(*) AS total_crashes,
        SUM(CASE WHEN INJURIES_TOTAL > 0 THEN 1 ELSE 0 END) AS injury_crashes,
        ROUND(100.0 * SUM(CASE WHEN INJURIES_TOTAL > 0 THEN 1 ELSE 0 END) / COUNT(*), 2) AS injury_rate
    FROM CrashTable
    GROUP BY STREET_NAME
    HAVING COUNT(*) > 100
    ORDER BY injury_rate DESC
    LIMIT 5
""", engine)
q9

,STREET_NAME,total_crashes,injury_crashes,injury_rate
0,MARQUETTE DR,271,81,29.89
1,FIFTH AVE,299,82,27.42
2,FRANKLIN BLVD,146,39,26.71
3,CORCORAN PL,122,31,25.41
4,BEVERLY AVE,251,62,24.70


Q10: Most common crash type for each year

In [ ]:
q10 = pd.read_sql("""
    WITH yearly_crash AS (
        SELECT
            year,
            FIRST_CRASH_TYPE,
            COUNT(*) AS crash_count,
            RANK() OVER (PARTITION BY year ORDER BY COUNT(*) DESC) AS rnk
        FROM CrashTable
        GROUP BY year, FIRST_CRASH_TYPE
    )
    SELECT year, FIRST_CRASH_TYPE, crash_count
    FROM yearly_crash
    WHERE rnk = 1
    ORDER BY year
""", engine)
#first, calculate crash counts per year and crash type
#second, find max crash count per year
#third, filter for most common crash type
#finally, order results by year and then by crash_count in descending order
q10

,year,FIRST_CRASH_TYPE,crash_count
0,2020,PARKED MOTOR VEHICLE,23072
1,2021,PARKED MOTOR VEHICLE,27457
2,2022,PARKED MOTOR VEHICLE,25468
3,2023,PARKED MOTOR VEHICLE,25136
4,2024,PARKED MOTOR VEHICLE,24839
5,2025,PARKED MOTOR VEHICLE,24322
6,2026,REAR END,5577


Q11: Day of the week with the highest average crashes per hour

In [ ]:
q11 = pd.read_sql("""
    SELECT
        CRASH_DAY_OF_WEEK,
        ROUND(AVG(CRASHES_PER_HOUR), 2) AS avg_crashes_per_hour
    FROM (
        SELECT
            CRASH_DAY_OF_WEEK,
            CRASH_HOUR,
            COUNT(*) AS CRASHES_PER_HOUR
        FROM CrashTable
        GROUP BY CRASH_DAY_OF_WEEK, CRASH_HOUR
    )
    GROUP BY CRASH_DAY_OF_WEEK
    ORDER BY avg_crashes_per_hour DESC

""", engine)

q11

,CRASH_DAY_OF_WEEK,avg_crashes_per_hour
0,6,4455.17


Q12: High-risk time slots:

[Hours grouped into buckets (Morning, Afternoon, Evening, Night) followed by bucket with the highest injury crashes]

In [ ]:
q12 = pd.read_sql("""
    SELECT
        CASE
            WHEN CRASH_HOUR BETWEEN 6 AND 11 THEN 'Morning'
            WHEN CRASH_HOUR BETWEEN 12 AND 17 THEN 'Afternoon'
            WHEN CRASH_HOUR BETWEEN 18 AND 21 THEN 'Evening'
            ELSE 'Night'
        END AS time_slot,
        SUM(CASE WHEN INJURIES_TOTAL > 0 THEN 1 ELSE 0 END) AS injury_crashes
    FROM CrashTable
    GROUP BY time_slot
    ORDER BY injury_crashes DESC
""", engine)

q12

,time_slot,injury_crashes
0,Afternoon,40200
1,Morning,23862
2,Evening,20392
3,Night,19884


Q13: Top 3 contributing causes for each crash type

In [ ]:
q13 = pd.read_sql("""
    WITH cause_counts AS (
        SELECT
            FIRST_CRASH_TYPE,
            PRIM_CONTRIBUTORY_CAUSE,
            COUNT(*) AS cause_count,
            ROW_NUMBER() OVER (
                PARTITION BY FIRST_CRASH_TYPE
                ORDER BY COUNT(*) DESC
            ) AS rn
        FROM CrashTable
        GROUP BY FIRST_CRASH_TYPE, PRIM_CONTRIBUTORY_CAUSE
    )
    SELECT FIRST_CRASH_TYPE, PRIM_CONTRIBUTORY_CAUSE, cause_count
    FROM cause_counts
    WHERE rn <= 3
    ORDER BY FIRST_CRASH_TYPE, rn
""", engine)

q13

,FIRST_CRASH_TYPE,PRIM_CONTRIBUTORY_CAUSE,cause_count
0,ANGLE,FAILING TO YIELD RIGHT-OF-WAY,22827
1,ANGLE,UNABLE TO DETERMINE,22564
2,ANGLE,DISREGARDING TRAFFIC SIGNALS,9518
3,ANIMAL,ANIMAL,305
4,ANIMAL,UNABLE TO DETERMINE,151
5,ANIMAL,NOT APPLICABLE,24
6,FIXED OBJECT,UNABLE TO DETERMINE,14453
7,FIXED OBJECT,NOT APPLICABLE,2894
8,FIXED OBJECT,DRIVING SKILLS/KNOWLEDGE/EXPERIENCE,2371
9,HEAD ON,UNABLE TO DETERMINE,1895


Q14: Year-over-year growth rate of crashes

In [ ]:
q14 = pd.read_sql("""
    WITH yearly AS (
        SELECT year, COUNT(*) AS total_crashes
        FROM CrashTable
        GROUP BY year
    ),
    yearly_with_prev AS (
        SELECT
            year,
            total_crashes,
            LAG(total_crashes) OVER (ORDER BY year) AS prev_year_crashes
        FROM yearly
    )
    SELECT
        year,
        total_crashes,
        prev_year_crashes,
        ROUND(100.0 * (total_crashes - prev_year_crashes)
            / prev_year_crashes, 2) AS yoy_growth_pct
    FROM yearly_with_prev
    WHERE prev_year_crashes IS NOT NULL
    ORDER BY year
""", engine)

q14

,year,total_crashes,prev_year_crashes,yoy_growth_pct
0,2021,107937,91509,17.95
1,2022,107462,107937,-0.44
2,2023,109714,107462,2.10
3,2024,110853,109714,1.04
4,2025,107965,110853,-2.61
5,2026,25494,107965,-76.39


Q15: Top 10 crash hotspot zones

In [ ]:
q15 = pd.read_sql("""
    SELECT
        ROUND(LATITUDE, 2) AS zone_lat,
        ROUND(LONGITUDE, 2) AS zone_lon,
        COUNT(*) AS crash_count
    FROM CrashTable
    WHERE LATITUDE IS NOT NULL AND LONGITUDE IS NOT NULL
    GROUP BY zone_lat, zone_lon
    ORDER BY crash_count DESC
    LIMIT 10
    """, engine)
q15

,zone_lat,zone_lon,crash_count
0,41.89,-87.63,7633
1,41.88,-87.63,5552
2,41.89,-87.62,5550
3,41.90,-87.62,5049
4,41.88,-87.62,4272
5,41.88,-87.64,3951
6,41.90,-87.63,3935
7,41.99,-87.66,3392
8,41.89,-87.64,3158
9,41.87,-87.64,3111


In [ ]:
q1.to_csv("q1_dangerous_weather_crash.csv", index=False)
q2.to_csv("q2_top_injury_streets.csv", index=False)
q3.to_csv("q3_injury_percentage.csv", index=False)
q4.to_csv("q4_peak_crash_hour.csv", index=False)
q5.to_csv("q5_nighttime_causes.csv", index=False)
q6.to_csv("q6_daylight_vs_darkness.csv", index=False)
q7.to_csv("q7_traffic_control_device_injuries.csv", index=False)
q8.to_csv("q8_top_crash_locations.csv", index=False)
q9.to_csv("q9_high_injury_rate_streets.csv", index=False)
q10.to_csv("q10_crash_type_per_year.csv", index=False)
q11.to_csv("q11_busiest_day.csv", index=False)
q12.to_csv("q12_time_slots.csv", index=False)
q13.to_csv("q13_causes_per_crash_type.csv", index=False)
q14.to_csv("q14_yoy_growth.csv", index=False)
q15.to_csv("q15_hotspot_zones.csv", index=False)

print("All CSVs saved!")

All CSVs saved!
